# HGCTM Revision — Presence/Absence Sensitivity at Fixed K = 7

This notebook addresses the reviewer's concern that the mineral-family "counts" are numbers of recorded mineral species rather than abundance measurements and may therefore be affected by documentation intensity.

The same 1,237-deposit coordinate-valid cohort and the same 35 mineral families are used. The original family-count matrix is converted to binary presence/absence:

```python
X_binary = (X_counts > 0).astype(float)
```

Two binary formulations are fitted:

1. **Binary Dirichlet–Multinomial (binary-DM)**  
   The original M7b architecture and likelihood are retained, but every positive family count is replaced by one. This provides the closest architectural comparison with the submitted count model.

2. **Bernoulli HGCTM**  
   Each deposit contributes one present/absent observation for every mineral family. Topic-specific family probabilities are generated with a sigmoid link, and the deposit-level family probability is the mixed-membership average across topics. This is the statistically natural presence/absence formulation and gives every deposit the same number of likelihood terms.

## Run design

- Lithology sources: Macrostrat and GLiM
- Prior settings:
  - original: sigma_lithology = 0.30, sigma_age = 0.15
  - equal-high: sigma_lithology = 0.30, sigma_age = 0.30
- Seeds: 42, 123, and 7
- Likelihoods: binary-DM and Bernoulli

Total in full mode: **24 fits**.

## Comparisons

The notebook reports:

- topic alignment with binary LDA;
- stability across seeds;
- alignment with the corresponding count-based HGCTM run;
- top-family overlap;
- dominant-topic agreement with the count-based model;
- class-frequency-weighted lithology-to-age effect ratios;
- strict-resolved ratios excluding `other` and `unknown`;
- likelihood metrics appropriate to each binary formulation;
- whether low-prevalence Topic 7 remains sensitive.

The previous count-based prior-sensitivity output is optional but strongly recommended because it enables direct run-by-run comparisons.

Original experiment notebook SHA-256: `6900e1e53eb92730851c9bc35e87c4f2731722ed0deccb9bc403ba223a689f7b`

## Required Kaggle inputs

Attach these existing files:

1. `Hgctm_stage0.zip`
2. `HGCTM_Phase1B_GLiM_Audit_Outputs.zip`
3. `HGCTM_Prior_and_Lithology_Class_Sensitivity_K7_Outputs.zip`  
   This third file is optional for fitting, but required for the direct comparison with the corresponding count-based HGCTM runs.

No new manual data preparation is required.

In [ ]:
# Install only packages that are missing.

import importlib.util
import subprocess
import sys

required = {
    "pyro": "pyro-ppl",
    "sklearn": "scikit-learn",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}

missing = [
    package
    for module, package in required.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("Installing:", missing)
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", *missing
    ])
else:
    print("All required packages are available.")


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import random
import re
import shutil
import sys
import time
import warnings
import zipfile

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import pyro
import pyro.distributions as dist
from pyro.infer import SVI, Trace_ELBO
from pyro.optim import ClippedAdam
from torch.nn.functional import softmax

from scipy.optimize import linear_sum_assignment
from scipy.stats import spearmanr
from sklearn.decomposition import LatentDirichletAllocation

warnings.filterwarnings("ignore")

WORK = Path("/kaggle/working")
OUT = WORK / "hgctm_presence_absence_sensitivity"
RUNS_DIR = OUT / "runs"
TABLE_DIR = OUT / "tables"
FIG_DIR = OUT / "figures"

for folder in [OUT, RUNS_DIR, TABLE_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

# ---------------- EXECUTION MODE ----------------
#
# quick:
#   1 seed, 400 steps, both likelihoods. Code check only.
# screening:
#   1 seed, 2,500 steps, both likelihoods.
# full:
#   3 seeds, 5,000 steps, both likelihoods.
#
RUN_MODE = "full"

K = 7
MIN_MINERALS = 3
MIN_DEPOSITS_PER_FAMILY = 10

EXPECTED_MODEL_N = 1335
EXPECTED_INVALID_COORDINATES = 98
EXPECTED_COMMON_N = 1237
EXPECTED_F = 35

PRIMARY_SEED = 42
FULL_SEEDS = [42, 123, 7]

if RUN_MODE == "quick":
    ACTIVE_SEEDS = [42]
    N_STEPS = 400
    ANNEAL_END = 120
    PRINT_EVERY = 100
elif RUN_MODE == "screening":
    ACTIVE_SEEDS = [42]
    N_STEPS = 2500
    ANNEAL_END = 750
    PRINT_EVERY = 500
elif RUN_MODE == "full":
    ACTIVE_SEEDS = FULL_SEEDS
    N_STEPS = 5000
    ANNEAL_END = 1500
    PRINT_EVERY = 500
else:
    raise ValueError("RUN_MODE must be quick, screening, or full.")

LR = 0.005
RARE_CLASS_THRESHOLD = 20
BERNOULLI_WARM_SIGNAL_SCALE = 0.75
TOP_N_FAMILIES = 10

LIKELIHOODS = ["binary_dm", "bernoulli"]

AGE_LEVELS = [
    "Cenozoic",
    "Mesozoic",
    "Paleozoic",
    "Precambrian",
    "Unknown",
]

SOURCE_SPECS = {
    "Macrostrat": "lithology_A_class",
    "GLiM": "lithology_B_class",
}

PRIOR_SETTINGS = {
    "original": {
        "sigma_litho": 0.30,
        "sigma_age": 0.15,
    },
    "equal_high": {
        "sigma_litho": 0.30,
        "sigma_age": 0.30,
    },
}

print("Run mode:", RUN_MODE)
print("Likelihoods:", LIKELIHOODS)
print("Seeds:", ACTIVE_SEEDS)
print("Steps:", N_STEPS)
print(
    "Planned fits:",
    len(LIKELIHOODS)
    * len(SOURCE_SPECS)
    * len(PRIOR_SETTINGS)
    * len(ACTIVE_SEEDS),
)


## 1. Locate and extract the frozen inputs

In [ ]:
def extract_zip_once(zip_path, target):
    target = Path(target)
    marker = target / ".extracted_ok"

    if marker.exists():
        return target

    if target.exists():
        shutil.rmtree(target)
    target.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(target)

    marker.write_text(
        datetime.now(timezone.utc).isoformat(),
        encoding="utf-8",
    )
    return target


def find_folder_with(required_names, label):
    required_names = set(required_names)
    roots = [Path("/kaggle/input"), WORK]

    # Already extracted.
    for root in roots:
        if not root.exists():
            continue
        for first_name in required_names:
            for path in root.rglob(first_name):
                folder = path.parent
                present = {
                    item.name
                    for item in folder.iterdir()
                    if item.is_file()
                }
                if required_names.issubset(present):
                    return folder

    # ZIP archive.
    for root in roots:
        if not root.exists():
            continue
        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    basenames = {
                        Path(name).name
                        for name in archive.namelist()
                    }
                    if required_names.issubset(basenames):
                        safe_label = re.sub(
                            r"[^a-z0-9]+",
                            "_",
                            label.lower(),
                        ).strip("_")
                        return extract_zip_once(
                            zip_path,
                            WORK / f"{safe_label}_extracted",
                        )
            except zipfile.BadZipFile:
                continue

    raise FileNotFoundError(
        f"Could not locate {label}. Required files: "
        f"{sorted(required_names)}"
    )


STAGE0 = find_folder_with(
    {"X_family_copper.csv", "covariates.csv"},
    "HGCTM Stage 0",
)

PHASE1B = find_folder_with(
    {
        "covariates_lithology_A_macrostrat_original.csv",
        "covariates_lithology_B_glim_only.csv",
    },
    "HGCTM Phase 1B",
)

print("Stage 0:", STAGE0)
print("Phase 1B:", PHASE1B)


In [ ]:
def locate_count_reference_root():
    expected_run = "macrostrat__original__seed_42.npz"
    roots = [Path("/kaggle/input"), WORK]

    # Search direct/extracted folders.
    for root in roots:
        if not root.exists():
            continue
        matches = list(root.rglob(expected_run))
        if matches:
            return matches[0].parent

    # Search ZIP files, extract a matching one.
    for root in roots:
        if not root.exists():
            continue
        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    basenames = {
                        Path(name).name
                        for name in archive.namelist()
                    }
                    if expected_run in basenames:
                        digest = hashlib.sha256(
                            str(zip_path).encode("utf-8")
                        ).hexdigest()[:12]
                        target = WORK / f"count_reference_{digest}"
                        extract_zip_once(zip_path, target)
                        matches = list(target.rglob(expected_run))
                        if matches:
                            return matches[0].parent
            except zipfile.BadZipFile:
                continue

    return None


COUNT_RUNS_DIR = locate_count_reference_root()

if COUNT_RUNS_DIR is None:
    print(
        "Count-based reference output not found. "
        "Binary models will run, but direct count-model comparisons will be skipped."
    )
else:
    print("Count-based reference runs:", COUNT_RUNS_DIR)


In [ ]:
def import_resume_binary_runs():
    imported = 0
    roots = [Path("/kaggle/input")]

    for root in roots:
        if not root.exists():
            continue

        for zip_path in root.rglob("*.zip"):
            try:
                with zipfile.ZipFile(zip_path) as archive:
                    names = archive.namelist()
                    if any(
                        name.endswith("_run_summary.json")
                        and "presence_absence" in name.lower()
                        for name in names
                    ):
                        digest = hashlib.sha256(
                            str(zip_path).encode("utf-8")
                        ).hexdigest()[:12]
                        target = WORK / f"presence_resume_{digest}"
                        extract_zip_once(zip_path, target)
            except zipfile.BadZipFile:
                continue

    for root in list(Path("/kaggle/input").rglob("*")) + list(
        WORK.glob("presence_resume_*")
    ):
        if not root.is_dir():
            continue
        for summary_path in root.rglob("*_run_summary.json"):
            try:
                metadata = json.loads(
                    summary_path.read_text(encoding="utf-8")
                )
            except Exception:
                continue

            if metadata.get("analysis") != "presence_absence_sensitivity":
                continue

            stem = summary_path.name.replace(
                "_run_summary.json",
                "",
            )

            if (RUNS_DIR / summary_path.name).exists():
                continue

            for source_file in summary_path.parent.glob(f"{stem}*"):
                if source_file.is_file():
                    shutil.copy2(
                        source_file,
                        RUNS_DIR / source_file.name,
                    )
            imported += 1

    return imported


print("Imported resumable binary runs:", import_resume_binary_runs())


## 2. Reconstruct the original model matrix and the common coordinate-valid cohort

In [ ]:
def canonical_id(value):
    if pd.isna(value):
        return None

    text = str(value).strip()

    try:
        number = float(text)
        if math.isfinite(number) and number.is_integer():
            return str(int(number))
    except Exception:
        pass

    return text


def canonicalize_index(frame, label):
    frame = frame.copy()
    frame.index = [
        canonical_id(value)
        for value in frame.index
    ]
    frame.index.name = "Mindat_id"

    if frame.index.isna().any():
        raise ValueError(f"{label} contains missing IDs.")
    if frame.index.duplicated().any():
        duplicates = frame.index[
            frame.index.duplicated()
        ].tolist()
        raise ValueError(
            f"{label} contains duplicate IDs: {duplicates[:10]}"
        )

    return frame


def canonicalize_id_column(frame, label):
    frame = frame.copy()

    if "Mindat_id" not in frame.columns:
        raise KeyError(f"{label} has no Mindat_id column.")

    frame["Mindat_id"] = frame["Mindat_id"].map(
        canonical_id
    )

    if frame["Mindat_id"].isna().any():
        raise ValueError(f"{label} contains missing IDs.")
    if frame["Mindat_id"].duplicated().any():
        duplicates = frame.loc[
            frame["Mindat_id"].duplicated(keep=False),
            "Mindat_id",
        ].tolist()
        raise ValueError(
            f"{label} contains duplicate IDs: {duplicates[:10]}"
        )

    return frame


X_raw = canonicalize_index(
    pd.read_csv(
        STAGE0 / "X_family_copper.csv",
        index_col=0,
    ),
    "X_family_copper",
)

cov_stage0 = canonicalize_index(
    pd.read_csv(
        STAGE0 / "covariates.csv",
        index_col=0,
    ),
    "covariates",
)

source_a = canonicalize_id_column(
    pd.read_csv(
        PHASE1B
        / "covariates_lithology_A_macrostrat_original.csv"
    ),
    "Macrostrat",
)

source_b = canonicalize_id_column(
    pd.read_csv(
        PHASE1B
        / "covariates_lithology_B_glim_only.csv"
    ),
    "GLiM",
)

print("Raw family matrix:", X_raw.shape)
print("Stage 0 covariates:", cov_stage0.shape)


In [ ]:
# Original model filter: deposits with at least three mapped mineral counts.
model_mask = X_raw.sum(axis=1) >= MIN_MINERALS
X_pre_family = X_raw.loc[model_mask].copy()

# Original family filter: at least ten deposits in the 1,335-deposit set.
family_presence = (X_pre_family > 0).sum(axis=0)
family_keep = family_presence[
    family_presence >= MIN_DEPOSITS_PER_FAMILY
].index.tolist()

X_model = X_pre_family[family_keep].copy()

model_ids = [
    mindat_id
    for mindat_id in X_model.index
    if mindat_id in cov_stage0.index
]

X_model = X_model.loc[model_ids].copy()
cov_model = cov_stage0.loc[model_ids].copy()

if X_model.shape != (EXPECTED_MODEL_N, EXPECTED_F):
    raise ValueError(
        f"Expected ({EXPECTED_MODEL_N}, {EXPECTED_F}), "
        f"found {X_model.shape}."
    )

print("Reconstructed count matrix:", X_model.shape)


In [ ]:
def valid_coordinate(lat, lon):
    if pd.isna(lat) or pd.isna(lon):
        return False

    try:
        lat = float(lat)
        lon = float(lon)
    except Exception:
        return False

    if not (-90 <= lat <= 90 and -180 <= lon <= 180):
        return False

    if abs(lat) < 1e-12 and abs(lon) < 1e-12:
        return False

    return True


a_small = source_a[
    [
        "Mindat_id",
        "Latitude",
        "Longitude",
        "lithology_A_class",
        "lithology_A_status",
    ]
].copy()

b_small = source_b[
    [
        "Mindat_id",
        "lithology_B_class",
        "lithology_B_status",
        "glim_level1_code",
        "glim_level1_description",
    ]
].copy()

lithology_table = a_small.merge(
    b_small,
    on="Mindat_id",
    how="inner",
    validate="one_to_one",
)

lithology_table["coordinate_valid"] = lithology_table.apply(
    lambda row: valid_coordinate(
        row["Latitude"],
        row["Longitude"],
    ),
    axis=1,
)

excluded_invalid = lithology_table[
    ~lithology_table["coordinate_valid"]
].copy()

valid_ids_set = set(
    lithology_table.loc[
        lithology_table["coordinate_valid"],
        "Mindat_id",
    ]
)

common_ids = [
    mindat_id
    for mindat_id in X_model.index
    if mindat_id in valid_ids_set
]

X_counts = X_model.loc[common_ids].copy()
X_binary = (X_counts > 0).astype(np.float32)
cov_common = cov_model.loc[common_ids].copy()

lithology_common = (
    lithology_table
    .set_index("Mindat_id")
    .loc[common_ids]
    .copy()
)

if len(excluded_invalid) != EXPECTED_INVALID_COORDINATES:
    raise ValueError(
        f"Expected {EXPECTED_INVALID_COORDINATES} invalid coordinates, "
        f"found {len(excluded_invalid)}."
    )

if X_counts.shape != (EXPECTED_COMMON_N, EXPECTED_F):
    raise ValueError(
        f"Expected ({EXPECTED_COMMON_N}, {EXPECTED_F}), "
        f"found {X_counts.shape}."
    )

print("Common count matrix:", X_counts.shape)
print("Common binary matrix:", X_binary.shape)
print("Excluded invalid coordinates:", len(excluded_invalid))


## 3. Audit documentation depth before modelling

In [ ]:
audit = cov_common[
    [
        "Deposit_type",
        "age_category",
        "Latitude",
        "Longitude",
    ]
].copy()

audit["Mindat_id"] = audit.index
audit["mapped_species_count"] = X_counts.sum(axis=1)
audit["present_family_count"] = X_binary.sum(axis=1)

depth_summary = pd.DataFrame({
    "metric": [
        "mapped_species_count",
        "present_family_count",
    ],
    "mean": [
        audit["mapped_species_count"].mean(),
        audit["present_family_count"].mean(),
    ],
    "median": [
        audit["mapped_species_count"].median(),
        audit["present_family_count"].median(),
    ],
    "minimum": [
        audit["mapped_species_count"].min(),
        audit["present_family_count"].min(),
    ],
    "maximum": [
        audit["mapped_species_count"].max(),
        audit["present_family_count"].max(),
    ],
    "q25": [
        audit["mapped_species_count"].quantile(0.25),
        audit["present_family_count"].quantile(0.25),
    ],
    "q75": [
        audit["mapped_species_count"].quantile(0.75),
        audit["present_family_count"].quantile(0.75),
    ],
})

spearman = spearmanr(
    audit["mapped_species_count"],
    audit["present_family_count"],
)

by_type = (
    audit.groupby("Deposit_type")
    .agg(
        n=("Mindat_id", "size"),
        mapped_species_mean=("mapped_species_count", "mean"),
        mapped_species_median=("mapped_species_count", "median"),
        present_families_mean=("present_family_count", "mean"),
        present_families_median=("present_family_count", "median"),
    )
    .reset_index()
)

depth_summary.to_csv(
    TABLE_DIR / "documentation_depth_summary.csv",
    index=False,
)
by_type.to_csv(
    TABLE_DIR / "documentation_depth_by_deposit_type.csv",
    index=False,
)
audit.to_csv(
    TABLE_DIR / "deposit_documentation_depth.csv",
    index=False,
)

print(depth_summary.round(3).to_string(index=False))
print(
    "\nSpearman correlation between mapped species count "
    "and present-family richness:",
    round(float(spearman.statistic), 4),
)
print("\nBy deposit type:")
print(by_type.round(3).to_string(index=False))


## 4. Prepare model lithology classes

The raw labels are preserved. For model fitting only, the single GLiM `evaporite` record is merged into `other`, matching the previous prior-sensitivity experiment.

In [ ]:
cohort = cov_common[
    [
        "Deposit_type",
        "age_category",
        "Latitude",
        "Longitude",
        "age_midpoint_ma",
    ]
].copy()

cohort["Mindat_id"] = cohort.index

raw_source_columns = {
    "Macrostrat": "lithology_A_class",
    "GLiM": "lithology_B_class",
}

for source_name, raw_column in raw_source_columns.items():
    raw_values = (
        lithology_common[raw_column]
        .fillna("unknown")
        .astype(str)
        .str.strip()
        .str.lower()
    )

    cohort[f"{source_name}_raw_lithology"] = raw_values
    cohort[f"{source_name}_model_lithology"] = (
        raw_values.replace({"evaporite": "other"})
    )

cohort = cohort.reset_index(drop=True)

model_lithology_columns = {
    source_name: f"{source_name}_model_lithology"
    for source_name in SOURCE_SPECS
}

all_model_levels = sorted(
    set().union(
        *[
            set(cohort[column].dropna().unique())
            for column in model_lithology_columns.values()
        ]
    )
)

class_count_rows = []

for source_name, column in model_lithology_columns.items():
    counts = (
        cohort[column]
        .value_counts()
        .reindex(all_model_levels, fill_value=0)
    )

    for lithology_class, count in counts.items():
        class_count_rows.append({
            "source": source_name,
            "lithology_class": lithology_class,
            "count": int(count),
            "percent": 100 * count / len(cohort),
            "rare_under_20": bool(
                0 < count < RARE_CLASS_THRESHOLD
            ),
            "zero_count": bool(count == 0),
        })

class_counts = pd.DataFrame(class_count_rows)

cohort.to_csv(
    TABLE_DIR / "common_cohort_covariates.csv",
    index=False,
)
X_counts.to_csv(
    TABLE_DIR / "common_cohort_count_matrix.csv"
)
X_binary.to_csv(
    TABLE_DIR / "common_cohort_binary_matrix.csv"
)
class_counts.to_csv(
    TABLE_DIR / "lithology_class_counts.csv",
    index=False,
)

print("Model lithology levels:", all_model_levels)
print(
    class_counts.pivot(
        index="lithology_class",
        columns="source",
        values="count",
    ).fillna(0).astype(int).to_string()
)


## 5. Fixed encodings and LDA references

In [ ]:
lithology_to_index = {
    label: index
    for index, label in enumerate(all_model_levels)
}
age_to_index = {
    label: index
    for index, label in enumerate(AGE_LEVELS)
}

age_values = (
    cohort["age_category"]
    .fillna("Unknown")
    .astype(str)
    .str.strip()
)

unexpected_age = set(age_values.unique()) - set(AGE_LEVELS)
if unexpected_age:
    raise ValueError(
        f"Unexpected age labels: {sorted(unexpected_age)}"
    )

age_idx_np = age_values.map(
    age_to_index
).to_numpy(dtype=np.int64)

source_lithology_idx = {}
source_lithology_tensors = {}

for source_name, column in model_lithology_columns.items():
    indices = cohort[column].map(
        lithology_to_index
    ).to_numpy(dtype=np.int64)

    source_lithology_idx[source_name] = indices
    source_lithology_tensors[source_name] = torch.tensor(
        indices,
        dtype=torch.long,
    )

X_counts_np = X_counts.to_numpy(dtype=np.float32)
X_binary_np = X_binary.to_numpy(dtype=np.float32)

X_binary_tensor = torch.tensor(
    X_binary_np,
    dtype=torch.float32,
)
age_tensor = torch.tensor(
    age_idx_np,
    dtype=torch.long,
)

N, F = X_binary_np.shape
L = len(all_model_levels)
A = len(AGE_LEVELS)

print(f"N={N}, F={F}, K={K}, L={L}, A={A}")


In [ ]:
# Count LDA reference, fitted with the same settings as the original experiment.
lda_count = LatentDirichletAllocation(
    n_components=K,
    doc_topic_prior=0.5 / K,
    topic_word_prior=0.01,
    max_iter=300,
    random_state=42,
    learning_method="batch",
    n_jobs=-1,
)
lda_count.fit(X_counts_np.astype(np.float64))
phi_lda_count = (
    lda_count.components_
    / lda_count.components_.sum(axis=1, keepdims=True)
)

# Binary LDA reference.
lda_binary = LatentDirichletAllocation(
    n_components=K,
    doc_topic_prior=0.5 / K,
    topic_word_prior=0.01,
    max_iter=300,
    random_state=42,
    learning_method="batch",
    n_jobs=-1,
)
lda_binary.fit(X_binary_np.astype(np.float64))
phi_lda_binary = (
    lda_binary.components_
    / lda_binary.components_.sum(axis=1, keepdims=True)
)
theta_lda_binary = lda_binary.transform(
    X_binary_np.astype(np.float64)
)

np.save(OUT / "phi_lda_count.npy", phi_lda_count)
np.save(OUT / "phi_lda_binary.npy", phi_lda_binary)
np.save(OUT / "theta_lda_binary.npy", theta_lda_binary)

pd.DataFrame(
    phi_lda_count,
    columns=X_counts.columns,
).to_csv(
    TABLE_DIR / "phi_lda_count.csv",
    index=False,
)

pd.DataFrame(
    phi_lda_binary,
    columns=X_binary.columns,
).to_csv(
    TABLE_DIR / "phi_lda_binary.csv",
    index=False,
)

print("Count LDA and binary LDA fitted.")


## 6. Binary HGCTM model

The prevalence component is unchanged from M7b. The content component is likelihood-specific:

- binary-DM: softmax topic-family composition;
- Bernoulli: sigmoid topic-family presence probabilities.

The same sparse family gate and age-weight parameter are retained.

In [ ]:
class HGCTM_BinarySensitivity(nn.Module):
    def __init__(
        self,
        K,
        F,
        L,
        A,
        likelihood,
        sigma_mu=2.0,
        sigma_litho=0.3,
        sigma_age=0.15,
    ):
        super().__init__()

        if likelihood not in {"binary_dm", "bernoulli"}:
            raise ValueError(
                "likelihood must be binary_dm or bernoulli"
            )

        self.K = K
        self.F = F
        self.L = L
        self.A = A
        self.likelihood = likelihood
        self.sigma_mu = sigma_mu
        self.sigma_litho = sigma_litho
        self.sigma_age = sigma_age

    def model(self, X, litho_idx, age_idx):
        N_local, F_local = X.shape
        K_local = self.K

        mu = pyro.sample(
            "mu",
            dist.Normal(
                torch.zeros(K_local, F_local),
                self.sigma_mu
                * torch.ones(K_local, F_local),
            ).to_event(2),
        )

        delta_litho = pyro.sample(
            "delta_litho",
            dist.Normal(
                torch.zeros(K_local, self.L, F_local),
                self.sigma_litho
                * torch.ones(K_local, self.L, F_local),
            ).to_event(3),
        )

        delta_age = pyro.sample(
            "delta_age",
            dist.Normal(
                torch.zeros(K_local, self.A, F_local),
                self.sigma_age
                * torch.ones(K_local, self.A, F_local),
            ).to_event(3),
        )

        omega_a = pyro.sample(
            "omega_a",
            dist.Beta(2.0, 2.0),
        )

        tau = pyro.sample(
            "tau",
            dist.Beta(
                torch.ones(F_local),
                3.0 * torch.ones(F_local),
            ).to_event(1),
        )

        B = pyro.sample(
            "B",
            dist.Normal(
                torch.zeros(self.L + self.A, K_local - 1),
                torch.ones(self.L + self.A, K_local - 1),
            ).to_event(2),
        )

        intercept = pyro.sample(
            "intercept",
            dist.Normal(
                torch.zeros(K_local - 1),
                torch.ones(K_local - 1),
            ).to_event(1),
        )

        if self.likelihood == "binary_dm":
            log_kappa = pyro.sample(
                "log_kappa",
                dist.Normal(3.5, 1.0),
            )
            kappa = torch.exp(log_kappa)

        with pyro.plate("deposits", N_local):
            z_litho = torch.nn.functional.one_hot(
                litho_idx,
                self.L,
            ).float()

            z_age = torch.nn.functional.one_hot(
                age_idx,
                self.A,
            ).float()

            z = torch.cat(
                [z_litho, z_age],
                dim=1,
            )

            eta = z @ B + intercept
            eta_full = torch.cat(
                [eta, torch.zeros(N_local, 1)],
                dim=1,
            )
            theta = softmax(
                eta_full,
                dim=1,
            )

            litho_dev = delta_litho[
                :, litho_idx, :
            ].permute(1, 0, 2)

            age_dev = delta_age[
                :, age_idx, :
            ].permute(1, 0, 2)

            psi = (
                mu.unsqueeze(0)
                + tau.view(1, 1, -1)
                * litho_dev
                + omega_a
                * tau.view(1, 1, -1)
                * age_dev
            )

            if self.likelihood == "binary_dm":
                topic_family = softmax(
                    psi,
                    dim=2,
                )

                p = torch.einsum(
                    "nk,nkf->nf",
                    theta,
                    topic_family,
                )
                p = p.clamp(min=1e-8)
                p = p / p.sum(
                    dim=1,
                    keepdim=True,
                )

                n_i = X.sum(dim=1)
                alpha = kappa * p

                pyro.sample(
                    "obs",
                    dist.DirichletMultinomial(
                        total_count=n_i.int(),
                        concentration=alpha,
                    ),
                    obs=X,
                )

            else:
                topic_family = torch.sigmoid(psi)

                p = torch.einsum(
                    "nk,nkf->nf",
                    theta,
                    topic_family,
                )
                p = p.clamp(
                    min=1e-6,
                    max=1.0 - 1e-6,
                )

                pyro.sample(
                    "obs",
                    dist.Bernoulli(
                        probs=p
                    ).to_event(1),
                    obs=X,
                )

    def guide(self, X, litho_idx, age_idx):
        _, F_local = X.shape
        K_local = self.K

        mu_loc = pyro.param(
            "mu_loc",
            torch.randn(K_local, F_local) * 0.1,
        )
        mu_scale = pyro.param(
            "mu_scale",
            0.5 * torch.ones(K_local, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "mu",
            dist.Normal(
                mu_loc,
                mu_scale,
            ).to_event(2),
        )

        delta_litho_loc = pyro.param(
            "delta_litho_loc",
            torch.zeros(K_local, self.L, F_local),
        )
        delta_litho_scale = pyro.param(
            "delta_litho_scale",
            0.2 * torch.ones(K_local, self.L, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_litho",
            dist.Normal(
                delta_litho_loc,
                delta_litho_scale,
            ).to_event(3),
        )

        delta_age_loc = pyro.param(
            "delta_age_loc",
            torch.zeros(K_local, self.A, F_local),
        )
        delta_age_scale = pyro.param(
            "delta_age_scale",
            0.1 * torch.ones(K_local, self.A, F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "delta_age",
            dist.Normal(
                delta_age_loc,
                delta_age_scale,
            ).to_event(3),
        )

        omega_a_a = pyro.param(
            "omega_a_a",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        omega_a_b = pyro.param(
            "omega_a_b",
            torch.tensor(2.0),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "omega_a",
            dist.Beta(
                omega_a_a,
                omega_a_b,
            ),
        )

        tau_a = pyro.param(
            "tau_a",
            torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        tau_b = pyro.param(
            "tau_b",
            3.0 * torch.ones(F_local),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "tau",
            dist.Beta(
                tau_a,
                tau_b,
            ).to_event(1),
        )

        B_loc = pyro.param(
            "B_loc",
            torch.zeros(self.L + self.A, K_local - 1),
        )
        B_scale = pyro.param(
            "B_scale",
            0.5 * torch.ones(self.L + self.A, K_local - 1),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "B",
            dist.Normal(
                B_loc,
                B_scale,
            ).to_event(2),
        )

        intercept_loc = pyro.param(
            "intercept_loc",
            torch.zeros(K_local - 1),
        )
        intercept_scale = pyro.param(
            "intercept_scale",
            0.5 * torch.ones(K_local - 1),
            constraint=torch.distributions.constraints.positive,
        )
        pyro.sample(
            "intercept",
            dist.Normal(
                intercept_loc,
                intercept_scale,
            ).to_event(1),
        )

        if self.likelihood == "binary_dm":
            log_kappa_loc = pyro.param(
                "log_kappa_loc",
                torch.tensor(3.5),
            )
            log_kappa_scale = pyro.param(
                "log_kappa_scale",
                torch.tensor(0.5),
                constraint=torch.distributions.constraints.positive,
            )
            pyro.sample(
                "log_kappa",
                dist.Normal(
                    log_kappa_loc,
                    log_kappa_scale,
                ),
            )


## 7. Warm starts, fitting, and parameter extraction

In [ ]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    pyro.set_rng_seed(seed)


def logit(values):
    values = np.clip(
        values,
        1e-5,
        1.0 - 1e-5,
    )
    return np.log(
        values / (1.0 - values)
    )


def make_mu_warm_start(likelihood):
    if likelihood == "binary_dm":
        phi = np.clip(
            phi_lda_binary,
            1e-6,
            None,
        )
        mu_init = np.log(phi)
        mu_init -= mu_init.mean(
            axis=1,
            keepdims=True,
        )
        return torch.tensor(
            mu_init,
            dtype=torch.float32,
        )

    # Bernoulli initialization:
    # begin at the observed marginal family prevalence and add
    # the relative topic signal from binary LDA.
    marginal_prevalence = np.clip(
        X_binary_np.mean(axis=0),
        0.01,
        0.99,
    )

    log_phi = np.log(
        np.clip(
            phi_lda_binary,
            1e-6,
            None,
        )
    )

    topic_signal = (
        log_phi
        - log_phi.mean(
            axis=0,
            keepdims=True,
        )
    )

    mu_init = (
        logit(marginal_prevalence)[None, :]
        + BERNOULLI_WARM_SIGNAL_SCALE
        * topic_signal
    )

    return torch.tensor(
        mu_init,
        dtype=torch.float32,
    )


def fit_binary_model(
    model,
    X,
    lithology_tensor,
    age_tensor,
    seed,
):
    set_all_seeds(seed)
    pyro.clear_param_store()

    optimizer = ClippedAdam({
        "lr": LR,
        "betas": (0.9, 0.999),
    })

    svi = SVI(
        model.model,
        model.guide,
        optimizer,
        loss=Trace_ELBO(),
    )

    # Create all parameters.
    svi.step(
        X,
        lithology_tensor,
        age_tensor,
    )

    pyro.param("mu_loc").data.copy_(
        make_mu_warm_start(
            model.likelihood
        )
    )

    # Same sparsity annealing initialization as M7b.
    pyro.param("tau_a").data.copy_(
        50.0 * torch.ones(X.shape[1])
    )
    pyro.param("tau_b").data.copy_(
        torch.ones(X.shape[1])
    )

    losses = []

    for step in range(N_STEPS):
        loss = svi.step(
            X,
            lithology_tensor,
            age_tensor,
        )
        losses.append(float(loss))

        if step < ANNEAL_END:
            progress = step / ANNEAL_END
            minimum_tau_a = (
                50.0 * (1.0 - progress)
                + progress
            )
            with torch.no_grad():
                pyro.param("tau_a").data.clamp_(
                    min=minimum_tau_a
                )

        if (step + 1) % PRINT_EVERY == 0:
            tau_a = pyro.param(
                "tau_a"
            ).detach()
            tau_b = pyro.param(
                "tau_b"
            ).detach()
            tau_mean = (
                tau_a / (tau_a + tau_b)
            ).mean().item()

            print(
                f"    step {step+1:>5d} "
                f"loss={loss:>12.1f} "
                f"tau_mean={tau_mean:.3f}"
            )

    return losses


def extract_parameters(model):
    mu = pyro.param(
        "mu_loc"
    ).detach().cpu().numpy()

    delta_litho = pyro.param(
        "delta_litho_loc"
    ).detach().cpu().numpy()

    delta_age = pyro.param(
        "delta_age_loc"
    ).detach().cpu().numpy()

    B = pyro.param(
        "B_loc"
    ).detach().cpu().numpy()

    intercept = pyro.param(
        "intercept_loc"
    ).detach().cpu().numpy()

    tau_a = pyro.param(
        "tau_a"
    ).detach().cpu().numpy()
    tau_b = pyro.param(
        "tau_b"
    ).detach().cpu().numpy()
    tau = tau_a / (tau_a + tau_b)

    omega_a_a = float(
        pyro.param("omega_a_a").item()
    )
    omega_a_b = float(
        pyro.param("omega_a_b").item()
    )
    omega_a = omega_a_a / (
        omega_a_a + omega_a_b
    )

    if model.likelihood == "binary_dm":
        kappa = float(
            np.exp(
                pyro.param(
                    "log_kappa_loc"
                ).item()
            )
        )
        topic_profile_raw = np.exp(mu)
        topic_profile_raw /= topic_profile_raw.sum(
            axis=1,
            keepdims=True,
        )
        topic_profile_normalized = topic_profile_raw.copy()
    else:
        kappa = np.nan
        topic_profile_raw = 1.0 / (
            1.0 + np.exp(-mu)
        )
        topic_profile_normalized = (
            topic_profile_raw
            / topic_profile_raw.sum(
                axis=1,
                keepdims=True,
            )
        )

    return {
        "mu": mu,
        "delta_litho": delta_litho,
        "delta_age": delta_age,
        "B": B,
        "intercept": intercept,
        "tau": tau,
        "omega_a": omega_a,
        "kappa": kappa,
        "topic_profile_raw": topic_profile_raw,
        "topic_profile_normalized": topic_profile_normalized,
    }


## 8. Deterministic outputs, likelihoods, alignment, and effect metrics

In [ ]:
def deterministic_outputs(
    model,
    params,
    X,
    lithology_tensor,
    age_tensor,
):
    mu = torch.tensor(
        params["mu"],
        dtype=torch.float32,
    )
    delta_litho = torch.tensor(
        params["delta_litho"],
        dtype=torch.float32,
    )
    delta_age = torch.tensor(
        params["delta_age"],
        dtype=torch.float32,
    )
    B = torch.tensor(
        params["B"],
        dtype=torch.float32,
    )
    intercept = torch.tensor(
        params["intercept"],
        dtype=torch.float32,
    )
    tau = torch.tensor(
        params["tau"],
        dtype=torch.float32,
    )
    omega_a = torch.tensor(
        params["omega_a"],
        dtype=torch.float32,
    )

    N_local = X.shape[0]

    with torch.no_grad():
        z_litho = torch.nn.functional.one_hot(
            lithology_tensor,
            L,
        ).float()

        z_age = torch.nn.functional.one_hot(
            age_tensor,
            A,
        ).float()

        z = torch.cat(
            [z_litho, z_age],
            dim=1,
        )

        eta = z @ B + intercept
        eta_full = torch.cat(
            [eta, torch.zeros(N_local, 1)],
            dim=1,
        )
        theta = softmax(
            eta_full,
            dim=1,
        )

        litho_dev = delta_litho[
            :, lithology_tensor, :
        ].permute(1, 0, 2)

        age_dev = delta_age[
            :, age_tensor, :
        ].permute(1, 0, 2)

        psi = (
            mu.unsqueeze(0)
            + tau.view(1, 1, -1)
            * litho_dev
            + omega_a
            * tau.view(1, 1, -1)
            * age_dev
        )

        if model.likelihood == "binary_dm":
            topic_family = softmax(
                psi,
                dim=2,
            )
            p = torch.einsum(
                "nk,nkf->nf",
                theta,
                topic_family,
            )
            p = p.clamp(min=1e-8)
            p /= p.sum(
                dim=1,
                keepdim=True,
            )
        else:
            topic_family = torch.sigmoid(psi)
            p = torch.einsum(
                "nk,nkf->nf",
                theta,
                topic_family,
            )
            p = p.clamp(
                min=1e-6,
                max=1.0 - 1e-6,
            )

    return {
        "theta": theta.cpu().numpy(),
        "topic_family_context": (
            topic_family.cpu().numpy()
        ),
        "p": p.cpu().numpy(),
    }


def dirichlet_multinomial_log_prob(
    X,
    p,
    kappa,
):
    X_t = torch.tensor(
        X,
        dtype=torch.float64,
    )
    p_t = torch.tensor(
        p,
        dtype=torch.float64,
    )

    p_t = p_t.clamp(min=1e-12)
    p_t /= p_t.sum(
        dim=1,
        keepdim=True,
    )

    alpha = float(kappa) * p_t
    alpha0 = alpha.sum(dim=1)
    n_i = X_t.sum(dim=1)

    log_coefficient = (
        torch.lgamma(n_i + 1.0)
        - torch.lgamma(X_t + 1.0).sum(
            dim=1
        )
    )

    log_beta_ratio = (
        torch.lgamma(alpha0)
        - torch.lgamma(alpha0 + n_i)
        + (
            torch.lgamma(alpha + X_t)
            - torch.lgamma(alpha)
        ).sum(dim=1)
    )

    return (
        log_coefficient + log_beta_ratio
    ).cpu().numpy()


def bernoulli_log_prob(
    X,
    p,
):
    X_t = torch.tensor(
        X,
        dtype=torch.float64,
    )
    p_t = torch.tensor(
        p,
        dtype=torch.float64,
    ).clamp(
        min=1e-12,
        max=1.0 - 1e-12,
    )

    per_family = (
        X_t * torch.log(p_t)
        + (1.0 - X_t)
        * torch.log(1.0 - p_t)
    )

    return (
        per_family.sum(dim=1).cpu().numpy(),
        per_family.cpu().numpy(),
    )


def cosine_similarity(a, b):
    denominator = (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )

    if denominator == 0:
        return 0.0

    return float(
        np.dot(a, b) / denominator
    )


def align_topics_to_reference(
    run_profile,
    reference_profile,
):
    similarity = np.zeros((K, K))

    for run_topic in range(K):
        for reference_topic in range(K):
            similarity[
                run_topic,
                reference_topic,
            ] = cosine_similarity(
                run_profile[run_topic],
                reference_profile[
                    reference_topic
                ],
            )

    run_indices, reference_indices = (
        linear_sum_assignment(-similarity)
    )

    reference_to_run = np.full(
        K,
        -1,
        dtype=int,
    )

    for run_topic, reference_topic in zip(
        run_indices,
        reference_indices,
    ):
        reference_to_run[
            reference_topic
        ] = run_topic

    topic_cosines = np.array([
        similarity[
            reference_to_run[reference_topic],
            reference_topic,
        ]
        for reference_topic in range(K)
    ])

    return {
        "reference_to_run": reference_to_run,
        "aligned_profile": run_profile[
            reference_to_run
        ],
        "topic_cosines": topic_cosines,
        "similarity_matrix": similarity,
    }


def top_family_jaccard(
    profile_1,
    profile_2,
    top_n=10,
):
    values = []

    for topic in range(K):
        top_1 = set(
            np.argsort(
                profile_1[topic]
            )[::-1][:top_n]
        )
        top_2 = set(
            np.argsort(
                profile_2[topic]
            )[::-1][:top_n]
        )

        values.append(
            len(top_1 & top_2)
            / len(top_1 | top_2)
        )

    return np.array(values)


def weighted_average(
    values,
    counts,
    include_mask,
):
    values = np.asarray(
        values,
        dtype=float,
    )
    counts = np.asarray(
        counts,
        dtype=float,
    )
    include_mask = np.asarray(
        include_mask,
        dtype=bool,
    )

    usable = (
        include_mask
        & (counts > 0)
        & np.isfinite(values)
    )

    if not usable.any():
        return np.nan

    return float(
        np.average(
            values[usable],
            weights=counts[usable],
        )
    )


def compute_effect_metrics(
    params,
    lithology_indices,
    age_indices,
):
    tau = params["tau"]

    lithology_class_effects = np.mean(
        np.abs(params["delta_litho"])
        * tau[None, None, :],
        axis=(0, 2),
    )

    age_class_effects = np.mean(
        np.abs(params["delta_age"])
        * tau[None, None, :]
        * params["omega_a"],
        axis=(0, 2),
    )

    lithology_counts = np.bincount(
        lithology_indices,
        minlength=L,
    )
    age_counts = np.bincount(
        age_indices,
        minlength=A,
    )

    lithology_nonmissing = np.array([
        label != "unknown"
        for label in all_model_levels
    ])
    age_nonmissing = np.array([
        label != "Unknown"
        for label in AGE_LEVELS
    ])

    lithology_strict = np.array([
        label not in {"other", "unknown"}
        for label in all_model_levels
    ])
    age_strict = age_nonmissing.copy()

    lithology_nonmissing_effect = weighted_average(
        lithology_class_effects,
        lithology_counts,
        lithology_nonmissing,
    )
    age_nonmissing_effect = weighted_average(
        age_class_effects,
        age_counts,
        age_nonmissing,
    )

    lithology_strict_effect = weighted_average(
        lithology_class_effects,
        lithology_counts,
        lithology_strict,
    )
    age_strict_effect = weighted_average(
        age_class_effects,
        age_counts,
        age_strict,
    )

    def safe_ratio(numerator, denominator):
        if (
            not np.isfinite(numerator)
            or not np.isfinite(denominator)
            or denominator <= 0
        ):
            return np.nan
        return float(numerator / denominator)

    metrics = {
        "lithology_effect_weighted_nonmissing":
            lithology_nonmissing_effect,
        "age_effect_weighted_nonmissing":
            age_nonmissing_effect,
        "ratio_weighted_nonmissing_primary":
            safe_ratio(
                lithology_nonmissing_effect,
                age_nonmissing_effect,
            ),
        "lithology_effect_weighted_strict":
            lithology_strict_effect,
        "age_effect_weighted_strict":
            age_strict_effect,
        "ratio_weighted_strict":
            safe_ratio(
                lithology_strict_effect,
                age_strict_effect,
            ),
        "tau_mean": float(
            np.mean(tau)
        ),
        "tau_median": float(
            np.median(tau)
        ),
    }

    detail_rows = []

    for index, label in enumerate(
        all_model_levels
    ):
        detail_rows.append({
            "covariate": "lithology",
            "class_index": index,
            "class_label": label,
            "count": int(
                lithology_counts[index]
            ),
            "class_effect": float(
                lithology_class_effects[index]
            ),
            "strict_resolved": (
                label not in {"other", "unknown"}
            ),
        })

    for index, label in enumerate(
        AGE_LEVELS
    ):
        detail_rows.append({
            "covariate": "age",
            "class_index": index,
            "class_label": label,
            "count": int(
                age_counts[index]
            ),
            "class_effect": float(
                age_class_effects[index]
            ),
            "strict_resolved": (
                label != "Unknown"
            ),
        })

    return metrics, pd.DataFrame(
        detail_rows
    )


## 9. Optional count-based reference loader

In [ ]:
def count_run_stem(
    source_name,
    prior_name,
    seed,
):
    return (
        f"{source_name.lower()}"
        f"__{prior_name}"
        f"__seed_{seed}"
    )


def load_count_reference(
    source_name,
    prior_name,
    seed,
):
    if COUNT_RUNS_DIR is None:
        return None

    stem = count_run_stem(
        source_name,
        prior_name,
        seed,
    )

    npz_path = COUNT_RUNS_DIR / f"{stem}.npz"
    summary_path = (
        COUNT_RUNS_DIR
        / f"{stem}_run_summary.json"
    )

    if not npz_path.exists():
        return None

    archive = np.load(
        npz_path,
        allow_pickle=True,
    )

    ids = [
        canonical_id(value)
        for value in archive["Mindat_id"].tolist()
    ]

    id_to_position = {
        mindat_id: position
        for position, mindat_id in enumerate(ids)
    }

    missing_ids = [
        mindat_id
        for mindat_id in common_ids
        if mindat_id not in id_to_position
    ]

    if missing_ids:
        raise ValueError(
            "Count-reference run does not contain all common IDs. "
            f"First missing IDs: {missing_ids[:10]}"
        )

    order = np.array([
        id_to_position[mindat_id]
        for mindat_id in common_ids
    ])

    phi = archive[
        "phi_global_aligned"
    ]

    theta = archive["theta"]
    if theta.shape[0] == len(ids):
        theta = theta[order]

    summary = None
    if summary_path.exists():
        summary = json.loads(
            summary_path.read_text(
                encoding="utf-8"
            )
        )

    return {
        "phi": phi,
        "theta": theta,
        "summary": summary,
        "archive_path": str(npz_path),
    }


availability_rows = []

for source_name in SOURCE_SPECS:
    for prior_name in PRIOR_SETTINGS:
        for seed in ACTIVE_SEEDS:
            reference = load_count_reference(
                source_name,
                prior_name,
                seed,
            )
            availability_rows.append({
                "source": source_name,
                "prior_setting": prior_name,
                "seed": seed,
                "count_reference_available": (
                    reference is not None
                ),
            })

count_reference_availability = pd.DataFrame(
    availability_rows
)
count_reference_availability.to_csv(
    TABLE_DIR / "count_reference_availability.csv",
    index=False,
)

print(
    count_reference_availability.to_string(
        index=False
    )
)


## 10. Resumable run plan

In [ ]:
run_plan_rows = []

for likelihood in LIKELIHOODS:
    for source_name in SOURCE_SPECS:
        for prior_name, prior in PRIOR_SETTINGS.items():
            for seed in ACTIVE_SEEDS:
                run_plan_rows.append({
                    "likelihood": likelihood,
                    "source": source_name,
                    "prior_setting": prior_name,
                    "seed": seed,
                    "sigma_litho": prior[
                        "sigma_litho"
                    ],
                    "sigma_age": prior[
                        "sigma_age"
                    ],
                })

run_plan_df = pd.DataFrame(
    run_plan_rows
)
run_plan_df.to_csv(
    TABLE_DIR / "planned_runs.csv",
    index=False,
)

print(run_plan_df.to_string(index=False))


In [ ]:
def binary_run_stem(
    likelihood,
    source_name,
    prior_name,
    seed,
):
    return (
        f"presence_absence"
        f"__{likelihood}"
        f"__{source_name.lower()}"
        f"__{prior_name}"
        f"__seed_{seed}"
    )


def load_existing_binary_summary(
    run_stem,
):
    summary_path = (
        RUNS_DIR
        / f"{run_stem}_run_summary.json"
    )

    if not summary_path.exists():
        return None

    return json.loads(
        summary_path.read_text(
            encoding="utf-8"
        )
    )


def save_binary_run(
    run_stem,
    likelihood,
    source_name,
    prior_name,
    seed,
    losses,
    params,
    outputs,
    alignment_binary_lda,
    effect_metrics,
    effect_details,
    log_prob_per_document,
    per_family_log_prob,
    elapsed_seconds,
):
    count_reference = load_count_reference(
        source_name,
        prior_name,
        seed,
    )

    count_comparison = {
        "count_reference_available": False,
        "count_topic_cosine_mean": np.nan,
        "count_topic_cosine_min": np.nan,
        "count_top10_jaccard_mean": np.nan,
        "count_top10_jaccard_min": np.nan,
        "dominant_topic_agreement_with_count": np.nan,
        "mean_absolute_theta_difference_from_count": np.nan,
        "count_ratio_weighted_nonmissing_primary": np.nan,
        "count_ratio_weighted_strict": np.nan,
    }

    count_alignment = None
    binary_theta_aligned_to_count = None

    if count_reference is not None:
        count_alignment = align_topics_to_reference(
            params[
                "topic_profile_normalized"
            ],
            count_reference["phi"],
        )

        binary_theta_aligned_to_count = (
            outputs["theta"][
                :,
                count_alignment[
                    "reference_to_run"
                ],
            ]
        )

        count_theta = count_reference["theta"]

        top_jaccard = top_family_jaccard(
            count_alignment[
                "aligned_profile"
            ],
            count_reference["phi"],
            top_n=TOP_N_FAMILIES,
        )

        count_comparison.update({
            "count_reference_available": True,
            "count_topic_cosine_mean": float(
                count_alignment[
                    "topic_cosines"
                ].mean()
            ),
            "count_topic_cosine_min": float(
                count_alignment[
                    "topic_cosines"
                ].min()
            ),
            "count_top10_jaccard_mean": float(
                top_jaccard.mean()
            ),
            "count_top10_jaccard_min": float(
                top_jaccard.min()
            ),
            "dominant_topic_agreement_with_count": float(
                np.mean(
                    np.argmax(
                        binary_theta_aligned_to_count,
                        axis=1,
                    )
                    == np.argmax(
                        count_theta,
                        axis=1,
                    )
                )
            ),
            "mean_absolute_theta_difference_from_count": float(
                np.mean(
                    np.abs(
                        binary_theta_aligned_to_count
                        - count_theta
                    )
                )
            ),
        })

        if count_reference["summary"] is not None:
            count_comparison[
                "count_ratio_weighted_nonmissing_primary"
            ] = count_reference["summary"].get(
                "ratio_weighted_nonmissing_primary",
                np.nan,
            )
            count_comparison[
                "count_ratio_weighted_strict"
            ] = count_reference["summary"].get(
                "ratio_weighted_strict",
                np.nan,
            )

    np.savez_compressed(
        RUNS_DIR / f"{run_stem}.npz",
        analysis=np.array(
            "presence_absence_sensitivity"
        ),
        likelihood=np.array(likelihood),
        source=np.array(source_name),
        prior_setting=np.array(prior_name),
        seed=np.array(seed),
        Mindat_id=np.array(common_ids),
        family_names=np.array(
            X_binary.columns
        ),
        lithology_levels=np.array(
            all_model_levels
        ),
        age_levels=np.array(
            AGE_LEVELS
        ),
        losses=np.asarray(losses),
        mu=params["mu"],
        delta_litho=params[
            "delta_litho"
        ],
        delta_age=params[
            "delta_age"
        ],
        B=params["B"],
        intercept=params["intercept"],
        tau=params["tau"],
        omega_a=np.array(
            params["omega_a"]
        ),
        kappa=np.array(
            params["kappa"]
        ),
        topic_profile_raw=params[
            "topic_profile_raw"
        ],
        topic_profile_normalized=params[
            "topic_profile_normalized"
        ],
        topic_profile_aligned_to_binary_lda=(
            alignment_binary_lda[
                "aligned_profile"
            ]
        ),
        theta=outputs["theta"],
        fitted_binary_probabilities=outputs[
            "p"
        ],
        log_prob_per_document=(
            log_prob_per_document
        ),
        per_family_log_prob=(
            per_family_log_prob
            if per_family_log_prob is not None
            else np.empty((0, 0))
        ),
        binary_lda_reference_to_run=(
            alignment_binary_lda[
                "reference_to_run"
            ]
        ),
        binary_lda_topic_cosines=(
            alignment_binary_lda[
                "topic_cosines"
            ]
        ),
        count_reference_to_run=(
            count_alignment[
                "reference_to_run"
            ]
            if count_alignment is not None
            else np.full(K, -1)
        ),
        count_topic_cosines=(
            count_alignment[
                "topic_cosines"
            ]
            if count_alignment is not None
            else np.full(K, np.nan)
        ),
        theta_aligned_to_count=(
            binary_theta_aligned_to_count
            if binary_theta_aligned_to_count is not None
            else np.empty((0, 0))
        ),
    )

    pd.DataFrame({
        "step": np.arange(
            1,
            len(losses) + 1,
        ),
        "loss": losses,
        "elbo": -np.asarray(losses),
    }).to_csv(
        RUNS_DIR
        / f"{run_stem}_loss_trace.csv",
        index=False,
    )

    effect_details.to_csv(
        RUNS_DIR
        / f"{run_stem}_class_effects.csv",
        index=False,
    )

    richness = X_binary_np.sum(axis=1)

    summary = {
        "analysis": "presence_absence_sensitivity",
        "run_stem": run_stem,
        "likelihood": likelihood,
        "source": source_name,
        "prior_setting": prior_name,
        "seed": int(seed),
        "run_mode": RUN_MODE,
        "n_deposits": int(N),
        "n_families": int(F),
        "n_steps": int(len(losses)),
        "sigma_litho": float(
            PRIOR_SETTINGS[
                prior_name
            ]["sigma_litho"]
        ),
        "sigma_age": float(
            PRIOR_SETTINGS[
                prior_name
            ]["sigma_age"]
        ),
        "elapsed_seconds": float(
            elapsed_seconds
        ),
        "omega_a": float(
            params["omega_a"]
        ),
        "kappa": (
            float(params["kappa"])
            if np.isfinite(params["kappa"])
            else None
        ),
        "elbo_mean_last_100": float(
            -np.mean(losses[-100:])
        ),
        "elbo_mean_last_100_per_deposit": float(
            -np.mean(losses[-100:])
            / N
        ),
        "log_prob_total": float(
            np.sum(log_prob_per_document)
        ),
        "log_prob_mean_per_deposit": float(
            np.mean(log_prob_per_document)
        ),
        "log_prob_per_present_family": float(
            np.sum(log_prob_per_document)
            / max(float(richness.sum()), 1.0)
        ),
        "log_prob_per_binary_cell": float(
            np.sum(log_prob_per_document)
            / float(N * F)
        ),
        "mean_topic_cosine_to_binary_lda": float(
            alignment_binary_lda[
                "topic_cosines"
            ].mean()
        ),
        "minimum_topic_cosine_to_binary_lda": float(
            alignment_binary_lda[
                "topic_cosines"
            ].min()
        ),
        **{
            key: (
                float(value)
                if np.isfinite(value)
                else None
            )
            for key, value
            in effect_metrics.items()
        },
        **{
            key: (
                float(value)
                if isinstance(
                    value,
                    (int, float, np.floating),
                )
                and np.isfinite(value)
                else bool(value)
                if isinstance(value, (bool, np.bool_))
                else None
            )
            for key, value
            in count_comparison.items()
        },
    }

    with open(
        RUNS_DIR
        / f"{run_stem}_run_summary.json",
        "w",
        encoding="utf-8",
    ) as handle:
        json.dump(
            summary,
            handle,
            indent=2,
            ensure_ascii=False,
        )

    return summary


## 11. Fit all 24 binary models

In [ ]:
summaries = []

for likelihood in LIKELIHOODS:
    for source_name in SOURCE_SPECS:
        lithology_indices = (
            source_lithology_idx[
                source_name
            ]
        )
        lithology_tensor = (
            source_lithology_tensors[
                source_name
            ]
        )

        for prior_name, prior in PRIOR_SETTINGS.items():
            for seed in ACTIVE_SEEDS:
                run_stem = binary_run_stem(
                    likelihood,
                    source_name,
                    prior_name,
                    seed,
                )

                existing = (
                    load_existing_binary_summary(
                        run_stem
                    )
                )

                if existing is not None:
                    print(
                        "SKIP completed run:",
                        run_stem,
                    )
                    summaries.append(existing)
                    continue

                print("\n" + "=" * 90)
                print(
                    f"LIKELIHOOD={likelihood} | "
                    f"SOURCE={source_name} | "
                    f"PRIOR={prior_name} | "
                    f"SEED={seed}"
                )
                print("=" * 90)

                model = HGCTM_BinarySensitivity(
                    K=K,
                    F=F,
                    L=L,
                    A=A,
                    likelihood=likelihood,
                    sigma_mu=2.0,
                    sigma_litho=prior[
                        "sigma_litho"
                    ],
                    sigma_age=prior[
                        "sigma_age"
                    ],
                )

                start = time.time()

                losses = fit_binary_model(
                    model=model,
                    X=X_binary_tensor,
                    lithology_tensor=(
                        lithology_tensor
                    ),
                    age_tensor=age_tensor,
                    seed=seed,
                )

                elapsed = time.time() - start

                params = extract_parameters(
                    model
                )

                outputs = deterministic_outputs(
                    model,
                    params,
                    X_binary_tensor,
                    lithology_tensor,
                    age_tensor,
                )

                if likelihood == "binary_dm":
                    log_prob_per_document = (
                        dirichlet_multinomial_log_prob(
                            X_binary_np,
                            outputs["p"],
                            params["kappa"],
                        )
                    )
                    per_family_log_prob = None
                else:
                    (
                        log_prob_per_document,
                        per_family_log_prob,
                    ) = bernoulli_log_prob(
                        X_binary_np,
                        outputs["p"],
                    )

                alignment_binary_lda = (
                    align_topics_to_reference(
                        params[
                            "topic_profile_normalized"
                        ],
                        phi_lda_binary,
                    )
                )

                (
                    effect_metrics,
                    effect_details,
                ) = compute_effect_metrics(
                    params,
                    lithology_indices,
                    age_idx_np,
                )

                summary = save_binary_run(
                    run_stem=run_stem,
                    likelihood=likelihood,
                    source_name=source_name,
                    prior_name=prior_name,
                    seed=seed,
                    losses=losses,
                    params=params,
                    outputs=outputs,
                    alignment_binary_lda=(
                        alignment_binary_lda
                    ),
                    effect_metrics=(
                        effect_metrics
                    ),
                    effect_details=(
                        effect_details
                    ),
                    log_prob_per_document=(
                        log_prob_per_document
                    ),
                    per_family_log_prob=(
                        per_family_log_prob
                    ),
                    elapsed_seconds=elapsed,
                )

                summaries.append(summary)

                print(
                    f"Completed {run_stem}: "
                    f"ratio="
                    f"{summary['ratio_weighted_nonmissing_primary']:.3f}, "
                    f"strict="
                    f"{summary['ratio_weighted_strict']:.3f}, "
                    f"binary-LDA cosine="
                    f"{summary['mean_topic_cosine_to_binary_lda']:.3f}"
                )

summary_df = pd.DataFrame(
    summaries
)

summary_df.to_csv(
    TABLE_DIR / "all_binary_run_summary.csv",
    index=False,
)

print("\nCompleted runs:", len(summary_df))
print(
    summary_df[
        [
            "likelihood",
            "source",
            "prior_setting",
            "seed",
            "ratio_weighted_nonmissing_primary",
            "ratio_weighted_strict",
            "mean_topic_cosine_to_binary_lda",
            "count_topic_cosine_mean",
            "dominant_topic_agreement_with_count",
        ]
    ].round(4).to_string(index=False)
)


## 12. Aggregate across seeds

In [ ]:
metrics_to_aggregate = [
    "ratio_weighted_nonmissing_primary",
    "ratio_weighted_strict",
    "lithology_effect_weighted_nonmissing",
    "age_effect_weighted_nonmissing",
    "omega_a",
    "kappa",
    "elbo_mean_last_100_per_deposit",
    "log_prob_mean_per_deposit",
    "log_prob_per_present_family",
    "log_prob_per_binary_cell",
    "mean_topic_cosine_to_binary_lda",
    "minimum_topic_cosine_to_binary_lda",
    "count_topic_cosine_mean",
    "count_topic_cosine_min",
    "count_top10_jaccard_mean",
    "count_top10_jaccard_min",
    "dominant_topic_agreement_with_count",
    "mean_absolute_theta_difference_from_count",
    "count_ratio_weighted_nonmissing_primary",
    "count_ratio_weighted_strict",
]

aggregate_rows = []

for (
    likelihood,
    source_name,
    prior_name,
), group in summary_df.groupby(
    [
        "likelihood",
        "source",
        "prior_setting",
    ],
    sort=True,
):
    row = {
        "likelihood": likelihood,
        "source": source_name,
        "prior_setting": prior_name,
        "n_seeds": int(len(group)),
    }

    for metric in metrics_to_aggregate:
        values = pd.to_numeric(
            group[metric],
            errors="coerce",
        )

        row[f"{metric}_mean"] = float(
            values.mean()
        )
        row[f"{metric}_median"] = float(
            values.median()
        )
        row[f"{metric}_min"] = float(
            values.min()
        )
        row[f"{metric}_max"] = float(
            values.max()
        )
        row[f"{metric}_std"] = float(
            values.std(ddof=0)
        )

    ratio = pd.to_numeric(
        group[
            "ratio_weighted_nonmissing_primary"
        ],
        errors="coerce",
    )
    strict_ratio = pd.to_numeric(
        group[
            "ratio_weighted_strict"
        ],
        errors="coerce",
    )

    row["primary_ratio_all_seeds_gt_1"] = bool(
        (ratio > 1).all()
    )
    row["strict_ratio_all_seeds_gt_1"] = bool(
        (strict_ratio > 1).all()
    )

    aggregate_rows.append(row)

aggregate_df = pd.DataFrame(
    aggregate_rows
)

aggregate_df.to_csv(
    TABLE_DIR
    / "binary_configuration_summary_across_seeds.csv",
    index=False,
)

display_columns = [
    "likelihood",
    "source",
    "prior_setting",
    "n_seeds",
    "ratio_weighted_nonmissing_primary_median",
    "ratio_weighted_nonmissing_primary_min",
    "ratio_weighted_nonmissing_primary_max",
    "ratio_weighted_strict_median",
    "mean_topic_cosine_to_binary_lda_median",
    "count_topic_cosine_mean_median",
    "dominant_topic_agreement_with_count_median",
    "primary_ratio_all_seeds_gt_1",
    "strict_ratio_all_seeds_gt_1",
]

print(
    aggregate_df[
        display_columns
    ].round(4).to_string(index=False)
)


## 13. Stability across seeds and comparison between the two binary likelihoods

In [ ]:
def load_binary_npz(
    likelihood,
    source_name,
    prior_name,
    seed,
):
    stem = binary_run_stem(
        likelihood,
        source_name,
        prior_name,
        seed,
    )
    path = RUNS_DIR / f"{stem}.npz"
    return np.load(
        path,
        allow_pickle=True,
    )


seed_stability_rows = []

for likelihood in LIKELIHOODS:
    for source_name in SOURCE_SPECS:
        for prior_name in PRIOR_SETTINGS:
            available = [
                seed
                for seed in ACTIVE_SEEDS
                if (
                    RUNS_DIR
                    / f"{binary_run_stem(likelihood, source_name, prior_name, seed)}.npz"
                ).exists()
            ]

            if not available:
                continue

            reference_seed = (
                PRIMARY_SEED
                if PRIMARY_SEED in available
                else available[0]
            )

            reference_profile = load_binary_npz(
                likelihood,
                source_name,
                prior_name,
                reference_seed,
            )[
                "topic_profile_aligned_to_binary_lda"
            ]

            for seed in available:
                current_profile = load_binary_npz(
                    likelihood,
                    source_name,
                    prior_name,
                    seed,
                )[
                    "topic_profile_aligned_to_binary_lda"
                ]

                for topic in range(K):
                    seed_stability_rows.append({
                        "likelihood": likelihood,
                        "source": source_name,
                        "prior_setting": prior_name,
                        "reference_seed": reference_seed,
                        "seed": seed,
                        "topic": topic + 1,
                        "cosine_to_reference_seed": (
                            cosine_similarity(
                                current_profile[topic],
                                reference_profile[topic],
                            )
                        ),
                    })

seed_stability = pd.DataFrame(
    seed_stability_rows
)
seed_stability.to_csv(
    TABLE_DIR / "within_binary_configuration_seed_stability.csv",
    index=False,
)


between_likelihood_rows = []

for source_name in SOURCE_SPECS:
    for prior_name in PRIOR_SETTINGS:
        for seed in ACTIVE_SEEDS:
            dm_path = (
                RUNS_DIR
                / f"{binary_run_stem('binary_dm', source_name, prior_name, seed)}.npz"
            )
            bern_path = (
                RUNS_DIR
                / f"{binary_run_stem('bernoulli', source_name, prior_name, seed)}.npz"
            )

            if not (
                dm_path.exists()
                and bern_path.exists()
            ):
                continue

            dm = np.load(
                dm_path,
                allow_pickle=True,
            )
            bern = np.load(
                bern_path,
                allow_pickle=True,
            )

            dm_profile = dm[
                "topic_profile_normalized"
            ]
            bern_profile = bern[
                "topic_profile_normalized"
            ]

            alignment = align_topics_to_reference(
                bern_profile,
                dm_profile,
            )

            bern_theta_aligned = bern[
                "theta"
            ][:, alignment["reference_to_run"]]

            dm_theta = dm["theta"]

            jaccard = top_family_jaccard(
                alignment["aligned_profile"],
                dm_profile,
                top_n=TOP_N_FAMILIES,
            )

            between_likelihood_rows.append({
                "source": source_name,
                "prior_setting": prior_name,
                "seed": seed,
                "mean_topic_cosine": float(
                    alignment[
                        "topic_cosines"
                    ].mean()
                ),
                "minimum_topic_cosine": float(
                    alignment[
                        "topic_cosines"
                    ].min()
                ),
                "mean_top10_jaccard": float(
                    jaccard.mean()
                ),
                "minimum_top10_jaccard": float(
                    jaccard.min()
                ),
                "dominant_topic_agreement": float(
                    np.mean(
                        np.argmax(
                            bern_theta_aligned,
                            axis=1,
                        )
                        == np.argmax(
                            dm_theta,
                            axis=1,
                        )
                    )
                ),
            })

between_likelihood_df = pd.DataFrame(
    between_likelihood_rows
)
between_likelihood_df.to_csv(
    TABLE_DIR / "between_binary_likelihood_comparison.csv",
    index=False,
)

print("Seed stability:")
print(
    seed_stability.groupby(
        [
            "likelihood",
            "source",
            "prior_setting",
        ]
    )["cosine_to_reference_seed"].agg(
        ["mean", "min", "std"]
    ).round(4).to_string()
)

print("\nBinary-DM versus Bernoulli:")
print(
    between_likelihood_df.round(4).to_string(
        index=False
    )
)


## 14. Topic-level count-versus-binary comparison and Topic 7 audit

In [ ]:
topic_comparison_rows = []
top_family_rows = []

for likelihood in LIKELIHOODS:
    for source_name in SOURCE_SPECS:
        for prior_name in PRIOR_SETTINGS:
            for seed in ACTIVE_SEEDS:
                stem = binary_run_stem(
                    likelihood,
                    source_name,
                    prior_name,
                    seed,
                )
                path = RUNS_DIR / f"{stem}.npz"

                if not path.exists():
                    continue

                binary_run = np.load(
                    path,
                    allow_pickle=True,
                )

                binary_profile = binary_run[
                    "topic_profile_normalized"
                ]

                count_reference = load_count_reference(
                    source_name,
                    prior_name,
                    seed,
                )

                if count_reference is None:
                    reference_profile = (
                        phi_lda_binary
                    )
                    reference_label = (
                        "binary_LDA"
                    )
                else:
                    reference_profile = (
                        count_reference["phi"]
                    )
                    reference_label = (
                        "count_HGCTM"
                    )

                alignment = align_topics_to_reference(
                    binary_profile,
                    reference_profile,
                )

                aligned_binary = alignment[
                    "aligned_profile"
                ]

                top_jaccard = top_family_jaccard(
                    aligned_binary,
                    reference_profile,
                    top_n=TOP_N_FAMILIES,
                )

                for topic in range(K):
                    topic_comparison_rows.append({
                        "likelihood": likelihood,
                        "source": source_name,
                        "prior_setting": prior_name,
                        "seed": seed,
                        "reference": reference_label,
                        "topic": topic + 1,
                        "topic_cosine": float(
                            alignment[
                                "topic_cosines"
                            ][topic]
                        ),
                        "top10_jaccard": float(
                            top_jaccard[topic]
                        ),
                        "is_topic_7": bool(
                            topic == 6
                        ),
                    })

                    top_indices = np.argsort(
                        aligned_binary[topic]
                    )[::-1][:TOP_N_FAMILIES]

                    for rank, family_index in enumerate(
                        top_indices,
                        start=1,
                    ):
                        top_family_rows.append({
                            "likelihood": likelihood,
                            "source": source_name,
                            "prior_setting": prior_name,
                            "seed": seed,
                            "topic": topic + 1,
                            "rank": rank,
                            "family": (
                                X_binary.columns[
                                    family_index
                                ]
                            ),
                            "profile_value": float(
                                aligned_binary[
                                    topic,
                                    family_index,
                                ]
                            ),
                        })

topic_comparison = pd.DataFrame(
    topic_comparison_rows
)
top_family_table = pd.DataFrame(
    top_family_rows
)

topic_comparison.to_csv(
    TABLE_DIR
    / "topic_level_count_binary_comparison.csv",
    index=False,
)
top_family_table.to_csv(
    TABLE_DIR
    / "binary_top_families_by_topic.csv",
    index=False,
)

topic7_summary = (
    topic_comparison[
        topic_comparison["is_topic_7"]
    ]
    .groupby(
        [
            "likelihood",
            "source",
            "prior_setting",
        ]
    )
    .agg(
        topic7_cosine_mean=(
            "topic_cosine",
            "mean",
        ),
        topic7_cosine_min=(
            "topic_cosine",
            "min",
        ),
        topic7_top10_jaccard_mean=(
            "top10_jaccard",
            "mean",
        ),
        topic7_top10_jaccard_min=(
            "top10_jaccard",
            "min",
        ),
    )
    .reset_index()
)

topic7_summary.to_csv(
    TABLE_DIR / "topic7_sensitivity_summary.csv",
    index=False,
)

print("Topic 7 sensitivity:")
print(
    topic7_summary.round(4).to_string(
        index=False
    )
)


## 15. Reviewer-oriented decision table

In [ ]:
decision_rows = []

for _, row in aggregate_df.iterrows():
    likelihood = row["likelihood"]
    source_name = row["source"]
    prior_name = row["prior_setting"]

    ratio_min = row[
        "ratio_weighted_nonmissing_primary_min"
    ]
    strict_min = row[
        "ratio_weighted_strict_min"
    ]
    seed_alignment = (
        seed_stability[
            (seed_stability["likelihood"] == likelihood)
            & (seed_stability["source"] == source_name)
            & (
                seed_stability[
                    "prior_setting"
                ]
                == prior_name
            )
        ][
            "cosine_to_reference_seed"
        ]
    )

    seed_alignment_mean = float(
        seed_alignment.mean()
    )
    seed_alignment_min = float(
        seed_alignment.min()
    )

    count_alignment = row[
        "count_topic_cosine_mean_median"
    ]
    dominant_agreement = row[
        "dominant_topic_agreement_with_count_median"
    ]

    if (
        ratio_min > 1
        and strict_min > 1
        and seed_alignment_mean >= 0.95
        and (
            not np.isfinite(count_alignment)
            or count_alignment >= 0.85
        )
    ):
        robustness_flag = "strong_binary_support"
    elif (
        ratio_min > 1
        and strict_min > 1
    ):
        robustness_flag = "effect_direction_robust_topics_partly_sensitive"
    else:
        robustness_flag = "documentation_depth_sensitivity_detected"

    decision_rows.append({
        "likelihood": likelihood,
        "source": source_name,
        "prior_setting": prior_name,
        "primary_ratio_min_across_seeds": ratio_min,
        "strict_ratio_min_across_seeds": strict_min,
        "seed_topic_cosine_mean": seed_alignment_mean,
        "seed_topic_cosine_min": seed_alignment_min,
        "count_topic_cosine_median": count_alignment,
        "dominant_topic_agreement_with_count_median": dominant_agreement,
        "robustness_flag": robustness_flag,
    })

decision_table = pd.DataFrame(
    decision_rows
)

decision_table.to_csv(
    TABLE_DIR
    / "reviewer_presence_absence_decision_table.csv",
    index=False,
)

print(decision_table.round(4).to_string(index=False))


## 16. Figures

In [ ]:
# Figure 1: documentation depth.
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    audit["mapped_species_count"],
    audit["present_family_count"],
    alpha=0.35,
)
ax.set_xlabel(
    "Mapped mineral-species count"
)
ax.set_ylabel(
    "Present mineral-family count"
)
ax.set_title(
    "Species-list depth versus binary family richness"
)
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "documentation_depth_species_vs_family_richness.png",
    dpi=300,
)
plt.close()


# Figure 2: effect ratio by run.
plot_df = summary_df.copy()
plot_df["label"] = (
    plot_df["likelihood"]
    + "\n"
    + plot_df["source"]
    + "\n"
    + plot_df["prior_setting"]
    + "\nseed "
    + plot_df["seed"].astype(str)
)

fig, ax = plt.subplots(figsize=(15, 6))
positions = np.arange(len(plot_df))
ax.scatter(
    positions,
    plot_df[
        "ratio_weighted_nonmissing_primary"
    ],
)
ax.axhline(
    1.0,
    linestyle="--",
    linewidth=1,
)
ax.set_xticks(positions)
ax.set_xticklabels(
    plot_df["label"],
    rotation=70,
    ha="right",
)
ax.set_ylabel(
    "Frequency-weighted lithology / age effect ratio"
)
ax.set_title(
    "Presence/absence sensitivity across likelihoods, sources, priors, and seeds"
)
plt.tight_layout()
plt.savefig(
    FIG_DIR
    / "presence_absence_effect_ratio_all_runs.png",
    dpi=300,
)
plt.close()


# Figure 3: count-versus-binary topic alignment.
comparison_plot = summary_df[
    summary_df[
        "count_reference_available"
    ].eq(True)
].copy()

if len(comparison_plot):
    fig, ax = plt.subplots(figsize=(12, 6))
    positions = np.arange(
        len(comparison_plot)
    )
    ax.scatter(
        positions,
        comparison_plot[
            "count_topic_cosine_mean"
        ],
        label="Mean topic cosine",
    )
    ax.scatter(
        positions,
        comparison_plot[
            "count_topic_cosine_min"
        ],
        label="Minimum topic cosine",
    )
    ax.set_ylim(0, 1.02)
    ax.set_xticks(positions)
    ax.set_xticklabels(
        (
            comparison_plot["likelihood"]
            + "\n"
            + comparison_plot["source"]
            + "\n"
            + comparison_plot["prior_setting"]
            + "\nseed "
            + comparison_plot["seed"].astype(str)
        ),
        rotation=70,
        ha="right",
    )
    ax.set_ylabel(
        "Cosine alignment with count-based HGCTM"
    )
    ax.set_title(
        "Count-versus-presence/absence topic alignment"
    )
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        FIG_DIR
        / "count_binary_topic_alignment.png",
        dpi=300,
    )
    plt.close()


# Figure 4: Topic 7 comparison.
if len(topic7_summary):
    fig, ax = plt.subplots(figsize=(10, 5))
    positions = np.arange(
        len(topic7_summary)
    )
    ax.scatter(
        positions,
        topic7_summary[
            "topic7_cosine_mean"
        ],
        label="Mean Topic 7 cosine",
    )
    ax.scatter(
        positions,
        topic7_summary[
            "topic7_cosine_min"
        ],
        label="Minimum Topic 7 cosine",
    )
    ax.set_ylim(0, 1.02)
    ax.set_xticks(positions)
    ax.set_xticklabels(
        (
            topic7_summary["likelihood"]
            + "\n"
            + topic7_summary["source"]
            + "\n"
            + topic7_summary["prior_setting"]
        ),
        rotation=45,
        ha="right",
    )
    ax.set_ylabel("Topic 7 cosine")
    ax.set_title(
        "Low-prevalence Topic 7 sensitivity"
    )
    ax.legend()
    plt.tight_layout()
    plt.savefig(
        FIG_DIR
        / "topic7_presence_absence_sensitivity.png",
        dpi=300,
    )
    plt.close()

print("Saved figures:")
for path in sorted(FIG_DIR.iterdir()):
    print(" -", path.name)


## 17. Provenance and downloadable archive

In [ ]:
manifest = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "analysis": "presence_absence_sensitivity",
    "notebook": "HGCTM_Presence_Absence_Sensitivity_K7.ipynb",
    "original_notebook_sha256": "6900e1e53eb92730851c9bc35e87c4f2731722ed0deccb9bc403ba223a689f7b",
    "run_mode": RUN_MODE,
    "inputs": {
        "stage0": str(STAGE0),
        "phase1b": str(PHASE1B),
        "count_reference_runs": (
            str(COUNT_RUNS_DIR)
            if COUNT_RUNS_DIR is not None
            else None
        ),
    },
    "cohort": {
        "original_model_n": int(X_model.shape[0]),
        "excluded_invalid_coordinates": int(len(excluded_invalid)),
        "common_valid_coordinate_n": int(N),
        "families": int(F),
    },
    "binary_transformation": "X_binary = (X_counts > 0).astype(float)",
    "likelihoods": {
        "binary_dm": (
            "Original sparse M7b architecture with the "
            "Dirichlet-Multinomial applied to binary family counts."
        ),
        "bernoulli": (
            "Independent family-level Bernoulli observations conditional "
            "on mixed-membership topic and context probabilities."
        ),
    },
    "sources": list(SOURCE_SPECS.keys()),
    "prior_settings": PRIOR_SETTINGS,
    "seeds": ACTIVE_SEEDS,
    "model": {
        "K": K,
        "sigma_mu": 2.0,
        "tau_prior": "Beta(1,3)",
        "omega_age_prior": "Beta(2,2)",
        "learning_rate": LR,
        "steps": N_STEPS,
        "anneal_end": ANNEAL_END,
        "LDA_random_state": 42,
        "bernoulli_warm_signal_scale": (
            BERNOULLI_WARM_SIGNAL_SCALE
        ),
    },
    "primary_effect_metric": (
        "Class-frequency-weighted nonmissing lithology effect divided by "
        "the corresponding nonmissing age effect."
    ),
    "strict_effect_metric": (
        "Same ratio after excluding lithology classes other and unknown."
    ),
}

with open(
    OUT / "run_manifest.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )

environment = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "torch": torch.__version__,
    "pyro": pyro.__version__,
}

with open(
    OUT / "environment_versions.json",
    "w",
    encoding="utf-8",
) as handle:
    json.dump(
        environment,
        handle,
        indent=2,
    )

readme = '''
HGCTM presence/absence sensitivity at fixed K=7

Purpose
-------
This analysis tests whether the reported assemblage modes and the
lithology-to-age effect depend on repeated mineral-species counts within
the 35 mineral families.

Data
----
The same 1,237 coordinate-valid deposits and 35 families are used.
Every positive family count is converted to one.

Models
------
1. Binary Dirichlet-Multinomial:
   closest architectural comparison with the submitted count model.

2. Bernoulli HGCTM:
   natural presence/absence formulation; every deposit contributes
   exactly 35 family-level Bernoulli observations.

Design
------
- Macrostrat and GLiM lithology
- original and equal-high prior settings
- three seeds
- K=7
- 24 fits in full mode

The optional prior-sensitivity output enables direct comparison with the
matching count-based run for topic alignment, top-family overlap, mixture
agreement, and effect-ratio change.
'''.strip()

(OUT / "README.txt").write_text(
    readme,
    encoding="utf-8",
)

archive_path = shutil.make_archive(
    str(
        WORK
        / "HGCTM_Presence_Absence_Sensitivity_K7_Outputs"
    ),
    "zip",
    root_dir=OUT,
)

print("Output archive:")
print(archive_path)

print("\nKey tables:")
for path in sorted(TABLE_DIR.iterdir()):
    print(" -", path.name)
